In [ ]:
import sys
sys.path.append('../')

In [ ]:
from parser import (remove_comments_and_docstrings,
                   tree_to_token_index,
                   index_to_code_token,
                   tree_to_variable_index)
from tree_sitter import Language, Parser

In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from preprocess import AST
from models import UniXcoder
import torch
device = torch.device("cuda:0")

In [ ]:
model = UniXcoder("microsoft/unixcoder-base").to(device)

In [ ]:
model

In [ ]:
# Encode maximum function
func_max = "def f(a,b): if a>b: return a else return b"
tokens_ids = model.tokenize([func_max],max_length=512,mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings,max_func_embedding = model(source_ids)

# Encode minimum function
func_min = "def f(a,b): if a<b: return a else return b"
tokens_ids = model.tokenize([func_min],max_length=512,mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings,min_func_embedding = model(source_ids)

# Encode NL
nl = "return maximum value"
tokens_ids = model.tokenize([nl],max_length=512,mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings,nl_embedding = model(source_ids)

print(max_func_embedding.shape)
print(max_func_embedding)

In [ ]:
a = torch.ones(2, 3, 4)
torch.nn.functional.normalize(a, p=2, dim=-1)

In [ ]:
torch.sqrt(torch.tensor(3))

In [ ]:
1 / 1.7321

In [ ]:
# Normalize embedding
norm_max_func_embedding = torch.nn.functional.normalize(max_func_embedding, p=2, dim=1)
norm_min_func_embedding = torch.nn.functional.normalize(min_func_embedding, p=2, dim=1)
norm_nl_embedding = torch.nn.functional.normalize(nl_embedding, p=2, dim=1)

max_func_nl_similarity = torch.einsum("ac,bc->ab",norm_max_func_embedding,norm_nl_embedding)
min_func_nl_similarity = torch.einsum("ac,bc->ab",norm_min_func_embedding,norm_nl_embedding)
max_func_min_func_similarity = torch.einsum("ac,bc->ab", norm_max_func_embedding, norm_min_func_embedding)

print(max_func_nl_similarity)
print(min_func_nl_similarity)
print(max_func_min_func_similarity)

In [ ]:
max_length = 512
max_ast = AST(func_max, 'python', model.tokenizer) # add spaces? 
tokens = max_ast[:max_length-4]
tokens = [model.tokenizer.cls_token, "<encoder-only>", model.tokenizer.sep_token] + tokens + [model.tokenizer.sep_token]
tokens_ids = model.tokenizer.convert_tokens_to_ids(tokens)
source_ids = torch.tensor(tokens_ids).unsqueeze(0).to(device)
tokens_embeddings, max_func_ast_embedding = model(source_ids)

max_length = 512
min_ast = AST(func_min, 'python', model.tokenizer) # add spaces? 
tokens = min_ast[:max_length-4]
tokens = [model.tokenizer.cls_token, "<encoder-only>", model.tokenizer.sep_token] + tokens + [model.tokenizer.sep_token]
tokens_ids = model.tokenizer.convert_tokens_to_ids(tokens)
source_ids = torch.tensor(tokens_ids).unsqueeze(0).to(device)
tokens_embeddings, min_func_ast_embedding = model(source_ids)

In [ ]:
model.tokenizer.tokenize(func_min)

In [ ]:
min_ast

In [ ]:
# Normalize embedding
norm_max_func_embedding = torch.nn.functional.normalize(max_func_embedding, p=2, dim=1)
norm_min_func_embedding = torch.nn.functional.normalize(min_func_embedding, p=2, dim=1)
norm_max_func_ast_embedding = torch.nn.functional.normalize(max_func_ast_embedding, p=2, dim=1)
norm_min_func_ast_embedding = torch.nn.functional.normalize(min_func_ast_embedding, p=2, dim=1)
norm_nl_embedding = torch.nn.functional.normalize(nl_embedding, p=2, dim=1)

max_func_nl_similarity = torch.einsum("ac,bc->ab",norm_max_func_embedding,norm_nl_embedding)
min_func_nl_similarity = torch.einsum("ac,bc->ab",norm_min_func_embedding,norm_nl_embedding)

print(max_func_nl_similarity)
print(min_func_nl_similarity)

In [ ]:
print('max func - max ast: {:.2f}, max func - min ast: {:.2f}, min func - max ast: {:.2f}, min func - min ast: {:.2f}, max ast - nl max: {:.2f}, min ast - nl max: {:.2f}'.format(
    torch.einsum("ac,bc->ab", norm_max_func_embedding, norm_max_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_max_func_embedding, norm_min_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_min_func_embedding, norm_max_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_min_func_embedding, norm_min_func_ast_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_max_func_ast_embedding, norm_nl_embedding).item(), 
    torch.einsum("ac,bc->ab", norm_min_func_ast_embedding, norm_nl_embedding).item()))